# NYC Taxi Lakehouse — Exploration Notebook

Scratchpad for investigating the Iceberg REST catalog, MinIO storage, and staging/marts data
directly with DuckDB — outside of dbt and Airflow, for ad hoc checks.

**Reminder on persistence:** every cell that does `ATTACH` below opens a fresh, isolated DuckDB
session. If a table shows up here, it's genuinely persisted in the REST catalog + MinIO, not a
DuckDB-memory artifact — that's the whole point of always re-attaching fresh rather than reusing
a long-lived connection.

## Setup — connect to MinIO + the Iceberg REST catalog

In [2]:
import duckdb

con = duckdb.connect()
con.sql("INSTALL iceberg; LOAD iceberg;")

con.sql("""
CREATE SECRET minio_s3 (
    TYPE s3,
    KEY_ID 'minioadmin',
    SECRET 'minioadmin',
    ENDPOINT 'minio:9000',
    URL_STYLE 'path',
    USE_SSL false
);
""")

con.sql("""
ATTACH 'open_lakehouse' AS raw_catalog (
    TYPE iceberg,
    ENDPOINT 'http://iceberg-rest:8181',
    AUTHORIZATION_TYPE 'none'
);
""")

print("Connected.")

Connected.


## What tables/schemas exist right now

In [3]:
con.sql("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,raw_catalog,main_staging,stg_yellow_taxi_trips,[__],[UNKNOWN],False
1,raw_catalog,open_lakehouse,yellow_taxi_trips,[__],[UNKNOWN],False


In [4]:
con.sql("""
    SELECT catalog_name, schema_name
    FROM information_schema.schemata
    WHERE catalog_name = 'raw_catalog'
""").df()

,catalog_name,schema_name
0,raw_catalog,main_staging
1,raw_catalog,open_lakehouse


## Inspect a table's schema

In [5]:
TABLE = "raw_catalog.main_staging.stg_yellow_taxi_trips" 

con.sql(f"DESCRIBE {TABLE}").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


## Row counts across layers

In [6]:
tables = {
    "raw": "raw_catalog.open_lakehouse.yellow_taxi_trips",
    "staging": "raw_catalog.main_staging.stg_yellow_taxi_trips",
    # "intermediate": "raw_catalog.main_intermediate.int_trips_enriched",
    # "marts": "raw_catalog.main_marts_core.fct_trips",
}

for label, tbl in tables.items():
    try:
        count = con.sql(f"SELECT COUNT(*) AS n FROM {tbl}").df().iloc[0]["n"]
        print(f"{label:12s} {tbl:55s} {count:>12,}")
    except Exception as e:
        print(f"{label:12s} {tbl:55s} ERROR: {e}")

raw          raw_catalog.open_lakehouse.yellow_taxi_trips              15,168,579
staging      raw_catalog.main_staging.stg_yellow_taxi_trips            15,168,109


## Sample rows

In [7]:
con.sql(f"SELECT * FROM {TABLE} LIMIT 10").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,year,month
0,2,2025-01-17 18:08:04,2025-01-17 18:19:55,<NA>,0.00,<NA>,None,161,186,0,...,0.5,0.0,0.0,1.0,17.97,NaN,NaN,0.75,<NA>,<NA>
1,2,2025-01-17 18:16:20,2025-01-17 18:33:22,<NA>,0.01,<NA>,None,234,143,0,...,0.5,0.0,0.0,1.0,25.84,NaN,NaN,0.75,<NA>,<NA>
2,1,2025-01-17 18:01:26,2025-01-17 18:22:49,<NA>,6.70,<NA>,None,231,141,0,...,0.5,0.0,0.0,1.0,30.45,NaN,NaN,0.75,<NA>,<NA>
3,2,2025-01-17 18:51:18,2025-01-17 19:11:39,<NA>,0.00,<NA>,None,234,239,0,...,0.5,0.0,0.0,1.0,6.37,NaN,NaN,0.75,<NA>,<NA>
4,2,2025-01-17 18:14:51,2025-01-17 18:25:51,<NA>,0.00,<NA>,None,193,179,0,...,0.5,0.0,0.0,1.0,11.54,NaN,NaN,0.00,<NA>,<NA>
5,2,2025-01-17 18:06:08,2025-01-17 18:11:01,<NA>,0.00,<NA>,None,164,164,0,...,0.5,0.0,0.0,1.0,12.86,NaN,NaN,0.75,<NA>,<NA>
6,1,2025-01-17 18:52:22,2025-01-17 18:58:24,<NA>,0.80,<NA>,None,114,113,0,...,0.5,0.0,0.0,1.0,10.78,NaN,NaN,0.75,<NA>,<NA>
7,2,2025-01-17 18:26:00,2025-01-17 18:54:38,<NA>,0.00,<NA>,None,74,209,0,...,0.5,0.0,0.0,1.0,37.37,NaN,NaN,0.75,<NA>,<NA>
8,2,2025-01-17 18:39:09,2025-01-17 18:39:25,<NA>,0.00,<NA>,None,88,88,0,...,0.5,0.0,0.0,1.0,3.00,NaN,NaN,0.75,<NA>,<NA>
9,2,2025-01-17 18:47:11,2025-01-17 19:06:21,<NA>,0.00,<NA>,None,261,114,0,...,0.5,0.0,0.0,1.0,21.63,NaN,NaN,0.75,<NA>,<NA>


## Data quality checks — fare/distance/tip ranges

In [11]:
con.sql(f"""
    SELECT
        min(fare_amount) AS min_fare, 
        max(fare_amount) AS max_fare,
        min(trip_distance) AS min_dist,
        max(trip_distance) AS max_dist,
        min(tip_amount) AS min_tip,
        max(tip_amount) AS max_tip,
        sum(CASE WHEN fare_amount <= 0 THEN 1 ELSE 0 END) AS zero_or_neg_fare_count,
        sum(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END) AS zero_dist_count,
        sum(CASE WHEN tip_amount < 0 THEN 1 ELSE 0 END) AS neg_tip_count
    FROM {TABLE}
""").df()

,min_fare,max_fare,min_dist,max_dist,min_tip,max_tip,zero_or_neg_fare_count,zero_dist_count,neg_tip_count
0,-1807.6,863372.12,0.0,386088.43,-220.0,525.0,728149.0,385824.0,509.0


## Timestamp sanity check (the pickup/dropoff issue from ingestion)

In [12]:
con.sql(f"""
    SELECT tpep_pickup_datetime, tpep_dropoff_datetime
    FROM {TABLE}
    WHERE tpep_pickup_datetime > tpep_dropoff_datetime
    LIMIT 20
""").df()

,tpep_pickup_datetime,tpep_dropoff_datetime


## Iceberg metadata — snapshots, manifests, partition spec

In [13]:
con.sql(f"SELECT * FROM iceberg_snapshots({TABLE})").df()

,sequence_number,snapshot_id,timestamp_ms,manifest_list,operation
0,1,1992353984052218448,2026-07-25 09:35:43.039,s3://warehouse/main_staging/stg_yellow_taxi_tr...,append


In [14]:
con.sql(f"SELECT * FROM iceberg_metadata({TABLE})").df()

,manifest_path,manifest_sequence_number,manifest_content,status,content,file_path,file_format,record_count
0,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,124917
1,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,124910
2,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,122880
3,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,122880
4,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,124927
...,...,...,...,...,...,...,...,...
111,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,114688
112,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,134648
113,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,124927
114,s3://warehouse/main_staging/stg_yellow_taxi_tr...,1,DATA,ADDED,EXISTING,s3://warehouse/main_staging/stg_yellow_taxi_tr...,parquet,124927


## Scratch area

Anything ad hoc goes below — new cells, exploratory queries, quick KPI checks against
`fct_trips` once it exists. Keep the setup cell at the top as the one source of truth for
connection config; don't duplicate ATTACH logic in scratch cells, just reuse `con`.

In [19]:
con.sql(f"""
    SELECT min(tpep_pickup_datetime), max(tpep_pickup_datetime) FROM {TABLE}
""").df()

,min(tpep_pickup_datetime),max(tpep_pickup_datetime)
0,2009-01-01 00:19:34,2025-05-01 00:48:13


In [22]:
con.sql(f"""
    select count(*)
    from {TABLE}
    where tpep_pickup_datetime < '2024-01-01'
""").df()

,count_star()
0,1


In [28]:
con.sql(f"""
SELECT * FROM raw_catalog.main_marts_core.dim_date LIMIT 10
""").df()

,date_key,year,month,day,day_name,is_weekend
0,2024-01-01,2024,1,1,Monday,False
1,2024-01-02,2024,1,2,Tuesday,False
2,2024-01-03,2024,1,3,Wednesday,False
3,2024-01-04,2024,1,4,Thursday,False
4,2024-01-05,2024,1,5,Friday,False
5,2024-01-06,2024,1,6,Saturday,True
6,2024-01-07,2024,1,7,Sunday,True
7,2024-01-08,2024,1,8,Monday,False
8,2024-01-09,2024,1,9,Tuesday,False
9,2024-01-10,2024,1,10,Wednesday,False


In [29]:
print(con.sql("SELECT COUNT(*) FROM raw_catalog.main_marts_core.fct_trips").df())
print(con.sql("SELECT COUNT(*) FROM raw_catalog.main_marts_core.dim_date").df())
print(con.sql("SELECT * FROM iceberg_metadata(raw_catalog.main_marts_core.fct_trips)").df())

   count_star()
0      14118204
   count_star()
0           487
                                        manifest_path  \
0   s3://warehouse/main_marts_core/fct_trips/metad...   
1   s3://warehouse/main_marts_core/fct_trips/metad...   
2   s3://warehouse/main_marts_core/fct_trips/metad...   
3   s3://warehouse/main_marts_core/fct_trips/metad...   
4   s3://warehouse/main_marts_core/fct_trips/metad...   
5   s3://warehouse/main_marts_core/fct_trips/metad...   
6   s3://warehouse/main_marts_core/fct_trips/metad...   
7   s3://warehouse/main_marts_core/fct_trips/metad...   
8   s3://warehouse/main_marts_core/fct_trips/metad...   
9   s3://warehouse/main_marts_core/fct_trips/metad...   
10  s3://warehouse/main_marts_core/fct_trips/metad...   
11  s3://warehouse/main_marts_core/fct_trips/metad...   
12  s3://warehouse/main_marts_core/fct_trips/metad...   
13  s3://warehouse/main_marts_core/fct_trips/metad...   
14  s3://warehouse/main_marts_core/fct_trips/metad...   
15  s3://warehouse/main_

In [30]:
con.sql("""
    SELECT
        pickup_hour,
        count(*) as trip_count,
        round(sum(fare_amount), 2) as total_revenue,
        round(avg(tip_pct) * 100, 1) as avg_tip_pct
    FROM raw_catalog.main_marts_core.fct_trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""").df()

,pickup_hour,trip_count,total_revenue,avg_tip_pct
0,0,420265,8045796.77,15.6
1,1,276842,4886489.42,15.4
2,2,180623,3065535.38,15.3
3,3,123064,2201350.66,14.6
4,4,92891,2115134.67,12.1
5,5,100049,2577841.76,12.5
6,6,207150,4559580.80,14.5
7,7,408276,7957527.09,16.1
8,8,563402,10259746.88,17.1
9,9,590332,10492414.37,18.7
